# Análise de dados públicos com as APIs do IBGE

Este notebook consulta dados públicos do IBGE e demonstra um fluxo completo de trabalho com APIs:

1. requisição HTTP;
2. validação da resposta;
3. transformação do JSON;
4. organização em DataFrames;
5. análise e visualização.

## Perguntas analisadas

- Como o PIB per capita de Brasil e Estados Unidos evoluiu ao longo do tempo?
- Quantas vezes o PIB per capita dos Estados Unidos superou o brasileiro em cada período?
- Como evoluiu o percentual de indivíduos com acesso à internet no Brasil?
- Quais são os nomes mais frequentes no país?


## 1. Bibliotecas e configuração

O projeto utiliza `requests` para acessar as APIs, `pandas` para organizar os dados e `Matplotlib` para criar gráficos.


In [ ]:
from __future__ import annotations

from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


## 2. Função para consultar a API

A função abaixo centraliza a chamada HTTP e evita repetir o mesmo tratamento em todas as análises.

Ela:

- define um limite de tempo para a requisição;
- verifica erros HTTP;
- tenta converter a resposta diretamente para JSON;
- apresenta uma mensagem útil caso o retorno não seja JSON válido.


In [ ]:
def obter_json(
    url: str,
    *,
    params: dict[str, Any] | None = None,
    timeout: int = 30,
) -> Any:
    """Consulta uma URL e devolve o conteúdo JSON validado."""

    try:
        resposta = requests.get(url, params=params, timeout=timeout)
        resposta.raise_for_status()
    except requests.RequestException as erro:
        raise RuntimeError(
            f"Não foi possível consultar a API. URL: {url}"
        ) from erro

    try:
        return resposta.json()
    except requests.JSONDecodeError as erro:
        previa = resposta.text[:200].replace("\n", " ")
        raise RuntimeError(
            "A API respondeu, mas o conteúdo não é um JSON válido. "
            f"Início da resposta: {previa!r}"
        ) from erro


## 3. Função para indicadores por país

A API Países devolve cada série histórica como uma lista de pequenos dicionários no formato `ano: valor`.

A função abaixo converte essa estrutura em um único DataFrame:

- anos no índice;
- um país em cada coluna;
- valores convertidos para número;
- linhas sem dados removidas.


In [ ]:
BASE_API_PAISES = "https://servicodados.ibge.gov.br/api/v1/paises"


def consultar_indicador_paises(
    paises: dict[str, str],
    indicador_id: str,
) -> pd.DataFrame:
    """Consulta um indicador para um ou mais países.

    Parameters
    ----------
    paises:
        Dicionário no formato {"código ISO": "nome exibido"}.
    indicador_id:
        Identificador numérico do indicador na API do IBGE.
    """

    codigos = "|".join(paises.keys())
    url = f"{BASE_API_PAISES}/{codigos}/indicadores/{indicador_id}"
    dados = obter_json(url)

    if not isinstance(dados, list) or not dados:
        raise ValueError("A API não devolveu uma lista de indicadores.")

    series = dados[0].get("series", [])

    if len(series) != len(paises):
        raise ValueError(
            "A quantidade de séries devolvida pela API é diferente "
            "da quantidade de países solicitada."
        )

    colunas: dict[str, pd.Series] = {}

    for nome_exibido, serie_pais in zip(paises.values(), series):
        valores_por_ano: dict[int, float] = {}

        for observacao in serie_pais.get("serie", []):
            if not observacao:
                continue

            ano_texto, valor_texto = next(iter(observacao.items()))
            ano = pd.to_numeric(ano_texto, errors="coerce")
            valor = pd.to_numeric(valor_texto, errors="coerce")

            if pd.notna(ano):
                valores_por_ano[int(ano)] = valor

        colunas[nome_exibido] = pd.Series(
            valores_por_ano,
            dtype="float64",
        )

    resultado = (
        pd.DataFrame(colunas)
        .sort_index()
        .dropna(how="all")
    )
    resultado.index.name = "ano"

    return resultado


## 4. PIB per capita: Brasil e Estados Unidos

O indicador `77823` representa o PIB per capita na API Países.

Além da comparação direta, será calculada a razão:

\[
\text{razão} = \frac{\text{PIB per capita dos EUA}}{\text{PIB per capita do Brasil}}
\]

Uma razão igual a 8, por exemplo, significa que o valor dos Estados Unidos foi oito vezes o valor brasileiro naquele ano.


In [ ]:
paises_pib = {
    "BR": "Brasil",
    "US": "Estados Unidos",
}

pib_per_capita = consultar_indicador_paises(
    paises=paises_pib,
    indicador_id="77823",
)

pib_per_capita["razao_eua_brasil"] = (
    pib_per_capita["Estados Unidos"] / pib_per_capita["Brasil"]
)

display(pib_per_capita.tail(10))


In [ ]:
pib_per_capita[["Brasil", "Estados Unidos"]].plot(
    figsize=(12, 6),
    marker="o",
)

plt.title("Evolução do PIB per capita")
plt.xlabel("Ano")
plt.ylabel("PIB per capita")
plt.legend(title="País")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
pib_per_capita["razao_eua_brasil"].plot(
    figsize=(12, 5),
    marker="o",
)

plt.title("Razão entre o PIB per capita dos EUA e do Brasil")
plt.xlabel("Ano")
plt.ylabel("Quantidade de vezes")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Leitura do resultado

O gráfico de razão permite observar não apenas a diferença absoluta entre os países, mas também como essa distância relativa mudou ao longo do tempo.

Os valores exatos dependem da atualização da API no momento da execução.


## 5. Indivíduos com acesso à internet no Brasil

O indicador `77857` representa o percentual de indivíduos com acesso à internet.


In [ ]:
acesso_internet = consultar_indicador_paises(
    paises={"BR": "Brasil"},
    indicador_id="77857",
).rename(columns={"Brasil": "percentual_com_acesso"})

display(acesso_internet.tail(10))


In [ ]:
acesso_internet["percentual_com_acesso"].plot(
    figsize=(12, 5),
    marker="o",
)

plt.title("Indivíduos com acesso à internet no Brasil")
plt.xlabel("Ano")
plt.ylabel("Percentual da população")
plt.ylim(bottom=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Leitura do resultado

A série permite acompanhar a expansão do acesso à internet no país e identificar períodos de crescimento mais acelerado ou estabilidade.


## 6. Ranking dos nomes mais frequentes

A API Nomes oferece um endpoint específico para o ranking nacional.

A resposta já possui os campos `nome`, `frequencia` e `ranking`, portanto pode ser convertida diretamente em DataFrame, sem criar um DataFrame separado para cada nome.


In [ ]:
URL_RANKING_NOMES = (
    "https://servicodados.ibge.gov.br/api/v2/censos/nomes/ranking"
)


def consultar_ranking_nomes(
    *,
    limite: int = 20,
    decada: int | None = None,
    sexo: str | None = None,
    localidade: str = "BR",
) -> pd.DataFrame:
    """Consulta o ranking de nomes mais frequentes do IBGE."""

    params: dict[str, Any] = {"localidade": localidade}

    if decada is not None:
        params["decada"] = decada
    if sexo is not None:
        params["sexo"] = sexo

    dados = obter_json(URL_RANKING_NOMES, params=params)

    if not isinstance(dados, list) or not dados:
        raise ValueError("A API não devolveu dados de ranking.")

    resultados = dados[0].get("res", [])
    ranking = pd.DataFrame(resultados)

    colunas_esperadas = ["ranking", "nome", "frequencia"]
    colunas_ausentes = set(colunas_esperadas) - set(ranking.columns)

    if colunas_ausentes:
        raise ValueError(
            f"Campos ausentes na resposta: {sorted(colunas_ausentes)}"
        )

    return (
        ranking[colunas_esperadas]
        .sort_values("ranking")
        .head(limite)
        .reset_index(drop=True)
    )


ranking_nomes = consultar_ranking_nomes(limite=20)
display(ranking_nomes)


In [ ]:
ranking_grafico = ranking_nomes.sort_values("frequencia")

plt.figure(figsize=(10, 7))
plt.barh(
    ranking_grafico["nome"].str.title(),
    ranking_grafico["frequencia"],
)

plt.title("20 nomes mais frequentes no Brasil")
plt.xlabel("Frequência")
plt.ylabel("Nome")
plt.tight_layout()
plt.show()


## 7. Resumo automático

A célula abaixo extrai alguns destaques dos dados obtidos na execução atual.


In [ ]:
ano_pib_mais_recente = pib_per_capita.dropna(
    subset=["Brasil", "Estados Unidos"]
).index.max()

ano_internet_mais_recente = acesso_internet.dropna().index.max()
nome_mais_frequente = ranking_nomes.iloc[0]

print(
    f"PIB per capita mais recente disponível: {ano_pib_mais_recente}."
)
print(
    "Razão EUA/Brasil nesse ano: "
    f"{pib_per_capita.loc[ano_pib_mais_recente, 'razao_eua_brasil']:.2f} vezes."
)
print(
    "Percentual mais recente de acesso à internet no Brasil "
    f"({ano_internet_mais_recente}): "
    f"{acesso_internet.loc[ano_internet_mais_recente, 'percentual_com_acesso']:.2f}%."
)
print(
    "Nome mais frequente no ranking: "
    f"{nome_mais_frequente['nome'].title()} "
    f"({int(nome_mais_frequente['frequencia']):,} registros)."
)


## 8. Conclusão

O projeto demonstra que uma análise com APIs não termina na requisição.

Um fluxo confiável também precisa:

- validar o status HTTP;
- conferir o formato da resposta;
- tratar valores ausentes e não numéricos;
- evitar código repetido;
- separar coleta, transformação e visualização;
- documentar limitações do serviço externo.

### Nota técnica

O material original tentava consultar um endpoint de projeções populacionais e recebia `JSONDecodeError`. Essa exceção ocorre quando o código tenta interpretar como JSON uma resposta que não contém JSON válido.

Como o endpoint de projeções não aparece no catálogo principal atual de APIs do IBGE, ele foi removido do fluxo executável deste notebook. O restante do projeto utiliza os serviços documentados de Países e Nomes.
